In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/hillstrom.db')

query = """
SELECT treatment, 
       AVG(recency) AS avg_recency, 
       AVG(history) AS avg_history,
       AVG(newbie) AS pct_newbie, 
       AVG(mens) AS pct_mens, 
       AVG(womens) AS pct_womens
FROM experiment 
GROUP BY treatment;
"""
summary_df = pd.read_sql(query, conn)
summary_df

,treatment,avg_recency,avg_history,pct_newbie,pct_mens,pct_womens
0,No E-Mail,5.749695,240.882653,0.501971,0.553224,0.547639
1,Womens E-Mail,5.767850,242.536633,0.503250,0.548932,0.550101


In [2]:
from scipy.stats import ttest_ind, chi2_contingency
from statsmodels.stats.multitest import multipletests

# Loading experiment data
df_exp = pd.read_sql(
    "SELECT * FROM experiment",
    conn
)

# Variables
continuous_vars = ['recency', 'history']
categorical_vars = ['newbie', 'mens', 'womens', 'channel', 'zip_code']

# Separate treatment and control
treatment = df_exp[df_exp['treatment'] == 'Womens E-Mail']
control = df_exp[df_exp['treatment'] == 'No E-Mail']

# Storing results
p_values = []
features = []
tests = []

# --------------------------------------------------
# Continuous variables: Welch's independent t-test
# --------------------------------------------------

for var in continuous_vars:
    stat, pval = ttest_ind(
        treatment[var],
        control[var],
        equal_var=False
    )

    p_values.append(pval)
    features.append(var)
    tests.append('Welch t-test')


# --------------------------------------------------
# Categorical variables: Chi-square test
# --------------------------------------------------

for var in categorical_vars:
    contingency_table = pd.crosstab(
        df_exp['treatment'],
        df_exp[var]
    )

    stat, pval, dof, expected = chi2_contingency(
        contingency_table
    )

    p_values.append(pval)
    features.append(var)
    tests.append('Chi-square')


# --------------------------------------------------
# Benjamini-Hochberg FDR correction
# --------------------------------------------------

alpha = 0.05

reject, pvals_corrected, _, _ = multipletests(
    p_values,
    alpha=alpha,
    method='fdr_bh'
)


# --------------------------------------------------
# Final balance table
# --------------------------------------------------

balance_table = pd.DataFrame({
    'Covariate': features,
    'Test': tests,
    'Raw p-value': p_values,
    'Corrected p-value (FDR)': pvals_corrected,
    'Significant Imbalance?': reject
})

balance_table

,Covariate,Test,Raw p-value,Corrected p-value (FDR),Significant Imbalance?
0,recency,Welch t-test,0.592545,0.908442,False
1,history,Welch t-test,0.501182,0.908442,False
2,newbie,Chi-square,0.799138,0.918822,False
3,mens,Chi-square,0.377813,0.908442,False
4,womens,Chi-square,0.616164,0.908442,False
5,channel,Chi-square,0.648887,0.908442,False
6,zip_code,Chi-square,0.918822,0.918822,False


After Benjamini–Hochberg FDR correction, none of the seven observed covariates show a statistically significant difference between the Womens E-Mail and No E-Mail groups (all corrected p-values > 0.05). Ergo, we find no evidence of covariate imbalance between the two treatment arms, consistent with the expected balance from a randomized controlled experiment.